## Settings

In [16]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [17]:
## libraries
import sys
import logging
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data,
    load_perturbed_data
)
from src.evaluators.perturbing import (
    eval_perturbed,
    find_perturbed_max,
    spec_marginal_delta,
    stat_perturbed_tost,
)
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initialization

In [ ]:
## reproducibility
N_REPEATS = 1
RANDOM_STATE = 42

## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_pert_all = load_perturbed_data()
    data_pert = {
        key: data_pert_all[key]
        for key in [
            "network_perturbed", 
            "invariants_perturbed", 
            "process_perturbed", 
            "signatures_perturbed"
        ]
        if key in data_pert_all
    }
finally:
    logging.disable(_disable)
    models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data_proc.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")
print("Perturbed Data:")
for json_key, methods in data_pert.items():
    print(f"  {json_key}:")
    for method, intense in methods.items():
        data = next(iter(intense.values()))
        n_r, n_c = data.shape
        print(f"   - {method}: {n_c} features, {n_r} observations")

Original Data: 32 features, 25 observations
Perturbed Data:
  network_perturbed:
   - densify: 24 features, 24 observations
   - rewire: 24 features, 24 observations
   - sample: 24 features, 24 observations
  invariants_perturbed:
   - jitter: 24 features, 24 observations
   - noise: 24 features, 24 observations
   - subset: 24 features, 24 observations
  process_perturbed:
   - bootstrapping: 10 features, 24 observations
   - scaling: 10 features, 24 observations
   - smoothing: 10 features, 24 observations
  signatures_perturbed:
   - jitter: 10 features, 24 observations
   - noise: 10 features, 24 observations
   - subset: 10 features, 24 observations


## Perturbation Evaluation
Four input perturbation methods (network, invariants, process, signatures) that each consist of three targeted techniques are applied while the target remains fixed. Model robustness is evaluated at maximum perturbation intensity using LOGO-CV (domain), where models are trained using original data and predict on the perturbations.

In [19]:
## perturbation evaluation
if "results_perturbed" not in globals():
    results_perturbed, _ = eval_perturbed(
        data = data,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
    )

## TOST Equivalence Tests
Each perturbation method is tested for equivalence against the unperturbed baseline EI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed EI values are paired on (model $\times$ domain), matching the falsification pipeline's pairing convention. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners on original results. Half the interquartile range of the baseline EI values sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## maximum intensity per perturbation method
results_perturbed_max = find_perturbed_max(
    results = results_perturbed
)

## empirical delta specification
delta = spec_marginal_delta(
    results = results_perturbed,
    feat_value = ["ei"],
    track = "frozen",
    method = "iqr",
    scale = 1.0,  ## iqr range
    decimals = 3
)

## tost equivalence test across all perturbation methods
results = stat_perturbed_tost(
    results = results_perturbed_max,
    feat_value = ["ei"],
    feat_group = ["perturbation", "method"],
    track = "frozen",
    delta = delta,
    decimals = 3
)

display(results)

TOST Equivalence (Wilcoxon Signed-Rank): n = 45, δ = 0.085
H₀: |Δ EI| ≥ δ
H₁: |Δ EI| < δ
Median Δ EI: Median of paired differences, not the difference of marginal medians
Rank-biserial r: Paired effect size, equivalence determined by TOST
TOST p: max(Upper p, Lower p)
Holm-adj. p: Holm-Bonferroni adjusted TOST p-value
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


Median Δ EI Rank-biserial r TOST p Holm-adj. p  \
Perturbation Method                                                         
network      densify             0.000          -0.395  0.000       0.000   
             rewire              0.000          -0.556  0.000       0.000   
             sample              0.000          -0.382  0.000       0.000   
invariants   jitter              0.000          -0.523  0.000       0.000   
             noise               0.000          -0.494  0.000       0.000   
             subset              0.000          -0.749  0.000       0.000   
process      bootstrapping      -0.003          -0.074  0.000       0.000   
             scaling             0.026           0.621  0.005       0.005   
             smoothing           0.004          -0.013  0.000       0.000   
signatures   jitter              0.007           0.544  0.000       0.000   
             noise               0.006           0.530  0.000       0.000   
             subset              0.000          -0.872  0.000       0.000   

                           Sig.  Eq.  
Perturbation Method                   
network      densify        ***  Yes  
             rewire         ***  Yes  
             sample         ***  Yes  
invariants   jitter         ***  Yes  
             noise          ***  Yes  
             subset         ***  Yes  
process      bootstrapping  ***  Yes  
             scaling         **  Yes  
             smoothing      ***  Yes  
signatures   jitter         ***  Yes  
             noise          ***  Yes  
             subset         ***  Yes

All four perturbation methods achieve TOST equivalence at maximum intensity. Effect sizes are negligible across all methods. The largest median shift (0.006, process) consumes less than one-quarter of the equivalence margin. EI is robust to aggressive corruption of network structure and its graph invariant vector representation, as well as stochastic process and its process signature vector representation.